# 🧠 Tarea 3 — Clasificación Multiclase con PyTorch
## Tabular Playground Series — May 2021 (Kaggle)

> **Disertación:** Este cuadernillo explica **paso a paso** cómo entrenar una red neuronal  
> profunda con PyTorch para clasificar 4 categorías de productos de e-commerce.

---

### 📚 ¿Qué aprenderemos aquí?

| Concepto | Fuente |
|---|---|
| `nn.Module` con capas expandidas + BatchNorm | Cuadernillo **00** |
| Búsqueda de hiperparámetros con `ray[tune]` | Cuadernillo **00** |
| `Dataset` + `DataLoader` con mini-batches | Cuadernillo **03** |
| Checkpoints automáticos del mejor modelo | Cuadernillo **04** |
| `ReduceLROnPlateau` + Early Stopping | Cuadernillo **02/04** |
| Visualización de arquitectura, métricas, convergencia | Todos |

---

### 📦 Dataset
- **Fuente:** [TPS May-2021 — Kaggle](https://www.kaggle.com/c/tabular-playground-series-may-2021/data)
- **Tamaño:** 200,000 muestras · 50 features numéricas · 4 clases
- **Problema:** Dataset sintético de e-commerce — features anónimas, sin nulos



---
## ⚙️ Paso 0 — Instalación de dependencias

Instalamos `ray[tune]` para la búsqueda de hiperparámetros (igual que en el cuadernillo 00)  
y `tqdm` para barras de progreso visuales.

> **¿Por qué `ray[tune]`?**  
> En vez de probar manualmente cada combinación de `lr`, `batch_size`, `activación`, etc.,  
> `ray.tune` hace un **grid search automático** y nos entrega la mejor configuración.


In [ ]:
!pip install -q "ray[tune]" tqdm

---
## 📦 Paso 1 — Importar librerías

Importamos:
- **PyTorch** (`torch`, `nn`, `optim`) → red neuronal
- **ray.tune** → búsqueda de hiperparámetros automática (cuadernillo 00)
- **tqdm** → barra de progreso visual por época
- **sklearn** → split, normalización, métricas
- **matplotlib / seaborn** → gráficas


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, log_loss)

from ray import tune

# ── Reproducibilidad ──────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Dispositivo: GPU si está disponible ──────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✔  Dispositivo activo: {device}')
print(f'✔  PyTorch version  : {torch.__version__}')

---
## 📂 Paso 2 — Cargar el dataset

Cargamos `train.csv` directamente desde Google Drive.  
Verificamos:
- ¿Cuántas filas/columnas hay?
- ¿Hay valores nulos?
- ¿Cómo se distribuyen las 4 clases?

> **Nota:** si el CSV ya está en el entorno local (Colab), descomenta la línea de Drive.


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# RUTA = '/content/drive/MyDrive/Universida/IA/Dataset/train.csv'
RUTA = './train.csv'   # ← ajusta si es necesario

df = pd.read_csv(RUTA)
print(f'Dimensiones : {df.shape}')
print(f'Nulos totales: {df.isnull().sum().sum()}')
print()
print('Distribución de clases:')
print(df['target'].value_counts())
df.head(3)

---
## 🔍 Paso 3 — Exploración visual (EDA)

Antes de entrenar, siempre exploramos los datos:
1. Distribución de las 4 clases
2. Distribución de las primeras features (para ver si hay outliers)

> **¿Por qué?**  
> Si las clases están desbalanceadas, el modelo aprende a predecir siempre la clase mayoritaria.  
> Si las features tienen escalas muy distintas, el gradiente diverge.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# — Distribución de clases —
conteo = df['target'].value_counts().sort_index()
colores = ['#4C9BE8', '#5DBB8A', '#F5A623', '#E05C5C']
bars = axes[0].bar(conteo.index, conteo.values, color=colores, edgecolor='white', width=0.6)
for bar, v in zip(bars, conteo.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 500,
                 f'{v:,}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Distribución de las 4 clases', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Clase'); axes[0].set_ylabel('Ejemplos')
axes[0].grid(axis='y', alpha=0.3)

# — Distribución de features 0–4 —
cols_feat = [f'feature_{i}' for i in range(5)]
for i, col in enumerate(cols_feat):
    axes[1].hist(df[col], bins=40, alpha=0.6, label=col)
axes[1].set_title('Distribución de las primeras 5 features', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Valor'); axes[1].set_ylabel('Frecuencia')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('TPS May-2021 — Exploración de Datos', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## 🔧 Paso 4 — Preprocesamiento

### 4.1 Balanceo de clases (Undersampling)

En el cuadernillo Lab4 original usamos undersampling: reducimos todas las clases  
al tamaño de la **clase minoritaria** para que el modelo no favorezca ninguna.

$$n_{\text{min}} = \min(|C_0|, |C_1|, |C_2|, |C_3|)$$

### 4.2 Normalización Z-score

Cada feature se normaliza con:
$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

> **Importante:** calculamos μ y σ **solo en train**, y los aplicamos en test.  
> Esto evita el **data leakage** (que el modelo "vea" información del test durante el entrenamiento).


In [ ]:
# ── 4.1 Undersampling ─────────────────────────────────────────
min_n  = df['target'].value_counts().min()
df_bal = pd.concat([
    resample(df[df['target'] == c], n_samples=min_n,
             replace=False, random_state=SEED)
    for c in df['target'].unique()
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Clase mínima      : {min_n:,}')
print(f'Total balanceado  : {len(df_bal):,}')
print(df_bal['target'].value_counts())

In [ ]:
# ── 4.2 Features, etiquetas y split ──────────────────────────
COLS = [f'feature_{i}' for i in range(50)]
le   = LabelEncoder()

X    = df_bal[COLS].values.astype(np.float32)
y    = le.fit_transform(df_bal['target'])        # clases → enteros 0/1/2/3

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

# ── Normalización Z-score ─────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)          # fit SOLO en train
X_test  = scaler.transform(X_test)               # transform en test

print(f'Train : {X_train.shape} | Test: {X_test.shape}')
print(f'Clases: {le.classes_}')
print(f'Post-normalización → media≈{X_train.mean():.3f}  std≈{X_train.std():.3f}')

### 4.3 ¿Qué hace la normalización Z-score? — Visualización

Mostramos la **misma feature** antes y después de normalizar para que quede claro visualmente.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Antes (valores originales)
raw_f0 = df_bal['feature_0'].values
axes[0].hist(raw_f0, bins=50, color='#E07B54', edgecolor='white')
axes[0].set_title(f'feature_0 ORIGINAL\nμ={raw_f0.mean():.2f}  σ={raw_f0.std():.2f}',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Valor'); axes[0].set_ylabel('Frecuencia')
axes[0].grid(alpha=0.3)

# Después (normalizado)
norm_f0 = X_train[:, 0]
axes[1].hist(norm_f0, bins=50, color='#4C9BE8', edgecolor='white')
axes[1].axvline(0, color='red', lw=2, linestyle='--', label='μ = 0')
axes[1].axvline(1, color='orange', lw=1.5, linestyle=':', label='σ = 1')
axes[1].axvline(-1, color='orange', lw=1.5, linestyle=':')
axes[1].set_title(f'feature_0 NORMALIZADA (Z-score)\nμ≈{norm_f0.mean():.3f}  σ≈{norm_f0.std():.3f}',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Valor'); axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Efecto de la Normalización Z-score en feature_0', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 4.4 Balance de clases — antes vs después del undersampling


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colores = ['#4C9BE8','#5DBB8A','#F5A623','#E05C5C']

orig = df['target'].value_counts().sort_index()
bal  = df_bal['target'].value_counts().sort_index()

for ax, datos, titulo in zip(axes, [orig, bal],
                              ['Distribución ORIGINAL', 'Distribución BALANCEADA']):
    bars = ax.bar(datos.index, datos.values, color=colores, edgecolor='white', width=0.6)
    for bar, v in zip(bars, datos.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 200,
                f'{v:,}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Clase'); ax.set_ylabel('Ejemplos')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Undersampling — Balanceo de Clases', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🗂 Paso 5 — `Dataset` y `DataLoader` (cuadernillo 03)

### ¿Por qué usar `Dataset` y `DataLoader`?

El cuadernillo **03** nos enseña que en vez de iterar manualmente los datos:
```python
# Forma manual (cuadernillo 00)
for idx in range(0, len(X_train), batch_size):
    X_batch = X_train[idx:idx+batch_size]
```

Es mejor usar la clase `Dataset` de PyTorch, que:
- Maneja el **shuffle** automáticamente cada época
- Permite **mini-batches** de forma eficiente
- Es **reutilizable** para train, val y predicción

```
Dataset ──► DataLoader ──► mini-batches automáticos ──► modelo
```


In [ ]:
class TPSDataset(Dataset):
    """
    Dataset tabular para TPS May-2021.
    Hereda de torch.utils.data.Dataset (cuadernillo 03).
    Implementa los tres métodos obligatorios:
      __init__  → guarda X e y como tensores
      __len__   → devuelve el número de muestras
      __getitem__→ devuelve un par (X[i], y[i])
    """
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)     # long para CrossEntropyLoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

# Instanciar datasets
ds_train = TPSDataset(X_train, y_train)
ds_test  = TPSDataset(X_test,  y_test)
print(f'Dataset train: {len(ds_train):,} muestras')
print(f'Dataset test : {len(ds_test):,} muestras')
print(f'Ejemplo [0]  : X={ds_train[0][0].shape}  y={ds_train[0][1]}')

In [ ]:
# DataLoader: empaqueta el Dataset en mini-batches
# shuffle=True en train para evitar que el modelo aprenda el orden
dl_train = DataLoader(ds_train, batch_size=512, shuffle=True,  num_workers=0, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=512, shuffle=False, num_workers=0)

print(f'Batches por época (train): {len(dl_train)}')
print(f'Batches por época (test) : {len(dl_test)}')

# Verificar un batch
xb, yb = next(iter(dl_train))
print(f'\nEjemplo de batch → X: {xb.shape}  y: {yb.shape}')

---
## 🏗 Paso 6 — Arquitectura con `nn.Module` (estilo cuadernillo 00)

### ¿Por qué `nn.Module` en vez de `Sequential`?

El cuadernillo **00** define el modelo como clase que hereda de `nn.Module`.  
Esto da **control total** del `forward` y permite:
- Capas con nombres descriptivos (`self.fc1`, `self.fc2batchnorm`...)
- Arquitecturas no lineales (residuales, ramas múltiples)
- Fácil acceso a parámetros individuales

### Arquitectura expandida (patrón del cuadernillo 00)

El modelo del cuadernillo 00 **expande** la dimensión hasta un máximo y luego **la reduce**:

```
Input(50) → starter(s) → s×2 → s×4 → s×2 → Output(4)
               ↑ BN+Act  ↑BN+Act ↑BN+Act ↑BN+Act
```

Donde `starter` es un hiperparámetro a buscar (16, 32, 64, 128, 256, 512).


In [ ]:
class ModelTPS(nn.Module):
    """
    Red neuronal tabular inspirada en el cuadernillo 00.
    Expande la dimensión hasta starter×4 y luego la reduce.
    
    Parámetros
    ----------
    input_dim  : número de features de entrada (50 para TPS)
    starter    : tamaño inicial de la capa oculta (hiperparámetro)
    n_classes  : número de clases de salida (4 para TPS)
    activation : función de activación (ReLU, LeakyReLU, ELU, SiLU...)
    dropout    : tasa de dropout para regularización
    """
    def __init__(self, input_dim=50, starter=64,
                 n_classes=4, activation=None, dropout=0.25):
        super().__init__()
        if activation is None:
            activation = nn.ReLU()
        self.activation = activation

        # — Bloque 1: input → starter —
        self.fc1      = nn.Linear(input_dim, starter)
        self.bn1      = nn.BatchNorm1d(starter)
        self.drop1    = nn.Dropout(dropout)

        # — Bloque 2: starter → starter×2 —
        self.fc2      = nn.Linear(starter, starter * 2)
        self.bn2      = nn.BatchNorm1d(starter * 2)
        self.drop2    = nn.Dropout(dropout)

        # — Bloque 3: starter×2 → starter×4 (expansión máxima) —
        self.fc3      = nn.Linear(starter * 2, starter * 4)
        self.bn3      = nn.BatchNorm1d(starter * 4)
        self.drop3    = nn.Dropout(dropout)

        # — Bloque 4: starter×4 → starter×2 (contracción) —
        self.fc4      = nn.Linear(starter * 4, starter * 2)
        self.bn4      = nn.BatchNorm1d(starter * 2)
        self.drop4    = nn.Dropout(dropout)

        # — Cabeza de salida —
        self.output   = nn.Linear(starter * 2, n_classes)

    def forward(self, x):
        # Bloque 1
        x = self.drop1(self.activation(self.bn1(self.fc1(x))))
        # Bloque 2
        x = self.drop2(self.activation(self.bn2(self.fc2(x))))
        # Bloque 3
        x = self.drop3(self.activation(self.bn3(self.fc3(x))))
        # Bloque 4
        x = self.drop4(self.activation(self.bn4(self.fc4(x))))
        # Salida (logits crudos → CrossEntropyLoss aplica Softmax)
        return self.output(x)

# Prueba rápida con starter=64
model_test = ModelTPS(input_dim=50, starter=64)
dummy = torch.randn(8, 50)
out   = model_test(dummy)
print(f'Output shape: {out.shape}  (esperado: [8, 4])')
total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Parámetros entrenables (starter=64): {total_params:,}')

### 6.2 Visualización de la arquitectura por capas

Dibujamos las capas del modelo con sus dimensiones para visualizar la expansión y contracción.


In [ ]:
def plot_architecture_tps(starter=64, input_dim=50, n_classes=4, title=None):
    """Visualiza la arquitectura expandida del cuadernillo 00."""
    dims   = [input_dim, starter, starter*2, starter*4, starter*2, n_classes]
    labels = [
        f'Input\n{input_dim} features',
        f'fc1 + BN\n{input_dim}→{starter}',
        f'fc2 + BN\n{starter}→{starter*2}',
        f'fc3 + BN\n{starter*2}→{starter*4}\n(máx. expansión)',
        f'fc4 + BN\n{starter*4}→{starter*2}',
        f'Output\n{n_classes} clases'
    ]
    colors = ['#78909C','#42A5F5','#5C6BC0','#AB47BC','#5C6BC0','#EF5350']

    n   = len(dims)
    fig, ax = plt.subplots(figsize=(n*2.2, 6))
    ax.set_xlim(-0.7, n-0.3); ax.set_ylim(-1.5, 5); ax.axis('off')
    mx = max(dims)

    for i, (d, lbl, col) in enumerate(zip(dims, labels, colors)):
        h = max(0.4, d / mx * 4)
        rect = mpatches.FancyBboxPatch(
            (i - 0.38, 2 - h/2), 0.76, h,
            boxstyle='round,pad=0.06', facecolor=col,
            alpha=0.88, edgecolor='white', lw=1.8)
        ax.add_patch(rect)
        ax.text(i, 2, str(d), ha='center', va='center',
                fontsize=10, fontweight='bold', color='white')
        ax.text(i, -0.2, lbl, ha='center', va='top',
                fontsize=8, color='#333', multialignment='center')
        if i < n-1:
            ax.annotate('', xy=(i+0.44, 2), xytext=(i+0.38, 2),
                        arrowprops=dict(arrowstyle='->', color='#555', lw=2))

    # Etiqueta de activación
    ax.text(n/2-0.5, 4.5,
            'Activación + Dropout en cada bloque oculto',
            ha='center', fontsize=10, color='#555', style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF9C4', alpha=0.8))

    if title is None:
        title = f'Arquitectura ModelTPS — starter={starter}'
    ax.set_title(title, fontsize=13, fontweight='bold', pad=20)
    plt.tight_layout(); plt.show()

# Visualizar con starter=64
plot_architecture_tps(starter=64)

### 6.3 Comparativa de arquitecturas con distintos `starter`

Mostramos cómo cambia la red al variar `starter`.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
starters = [32, 64, 128]
colores_s = ['#42A5F5','#AB47BC','#EF5350']

for ax, s, col in zip(axes, starters, colores_s):
    dims = [50, s, s*2, s*4, s*2, 4]
    x    = list(range(len(dims)))
    ax.plot(x, dims, 'o-', color=col, lw=2.5, markersize=8, markerfacecolor='white',
            markeredgewidth=2.5)
    ax.fill_between(x, dims, alpha=0.15, color=col)
    for xi, d in zip(x, dims):
        ax.text(xi, d + max(dims)*0.04, str(d), ha='center', fontsize=9, fontweight='bold')
    ax.set_title(f'starter = {s}  | params≈{50*s + s*s*2 + s*2*s*4 + s*4*s*2 + s*2*4:,}',
                 fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(['Input','fc1','fc2','fc3
(pico)','fc4','Out'], fontsize=8)
    ax.set_ylabel('Dimensión'); ax.grid(alpha=0.3)

plt.suptitle('Comparativa de arquitecturas — variando starter', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## ✅ Paso 7 — Función `evaluate` (equivalente a `test` del cuadernillo 00)

El cuadernillo **00** separa el cálculo del accuracy en una función independiente `test()`.  
La adaptamos para multiclase (4 clases) usando `DataLoader` para mayor eficiencia.

```python
# Cuadernillo 00 (original — una muestra a la vez)
for idx in range(len(X_test)):
    preds = model(X_test[idx].view(1, 10))

# Nuestra versión — DataLoader (por batches, más rápido)
for xb, yb in dl_test:
    preds = model(xb)
```


In [ ]:
def evaluate(model, dl, device='cpu'):
    """
    Calcula accuracy y log-loss sobre un DataLoader.
    Equivalente a la función test() del cuadernillo 00,
    mejorada para usar batches en vez de muestra a muestra.
    """
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(device)
            logits = model(xb)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())
    model.train()   # volver a modo entrenamiento (como en cuadernillo 00)

    acc  = accuracy_score(all_labels, all_preds)
    loss = log_loss(all_labels, all_probs)
    return acc, loss, np.array(all_preds), np.array(all_labels)

print('✔  Función evaluate() definida')

---
## 🔎 Paso 8 — Búsqueda de hiperparámetros con `ray.tune` (cuadernillo 00)

### ¿Qué es `ray.tune`?

En el cuadernillo **00**, en vez de elegir manualmente los hiperparámetros,  
se hace un **grid search** automático probando todas las combinaciones de:

| Hiperparámetro | Opciones a probar |
|---|---|
| `starter` (tamaño capa) | 32, 64, 128 |
| `activation` (función activación) | ReLU, LeakyReLU, ELU, SiLU |
| `optimizer` | Adam, AdamW |
| `lr` (learning rate) | 1e-3, 1e-4 |
| `criterion` (función de pérdida) | CrossEntropyLoss |

> ⚠️ **Para disertación:** usamos un grid pequeño para que sea rápido.  
> En producción (cuadernillo 00 completo) se pueden probar cientos de combinaciones.


In [ ]:
def train_tune(config):
    """
    Función de entrenamiento para ray.tune.
    Recibe 'config' con los hiperparámetros a probar.
    Reporta accuracy al tune para que encuentre el mejor config.
    """
    # ── Construir modelo con los hiperparámetros del config ──
    model = ModelTPS(
        input_dim  = 50,
        starter    = config['starter'],
        n_classes  = 4,
        activation = config['activation'],
        dropout    = config.get('dropout', 0.25)
    ).to(device)

    criterion = config['criterion']()
    optimizer = config['optimizer'](model.parameters(), lr=config['lr'])

    # ── Mini-loop de entrenamiento (épocas rápidas para el search) ──
    batch_size = config['batch_size']
    X_tr_t = torch.tensor(X_train, dtype=torch.float32)
    y_tr_t = torch.tensor(y_train, dtype=torch.long)

    model.train()
    for epoch in range(config['epochs']):
        # Mini-batches manuales (estilo cuadernillo 00)
        idxs = torch.randperm(len(X_tr_t))
        X_sh, y_sh = X_tr_t[idxs], y_tr_t[idxs]

        for i in range(0, len(X_sh), batch_size):
            xb = X_sh[i:i+batch_size].to(device)
            yb = y_sh[i:i+batch_size].to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    # ── Evaluar y reportar ──
    acc, val_loss, _, _ = evaluate(model, dl_test, device)
    tune.report(accuracy=acc, val_loss=val_loss)

print('✔  Función train_tune() definida — lista para ray.tune')

In [ ]:
# ── Configuración del grid search (igual que cuadernillo 00) ──
search_config = {
    'epochs'    : tune.grid_search([15]),                   # épocas rápidas para el search
    'batch_size': tune.grid_search([512]),
    'criterion' : tune.grid_search([nn.CrossEntropyLoss]),
    'starter'   : tune.grid_search([32, 64, 128]),
    'activation': tune.grid_search([nn.ReLU(), nn.LeakyReLU(), nn.ELU(), nn.SiLU()]),
    'optimizer' : tune.grid_search([optim.Adam, optim.AdamW]),
    'lr'        : tune.grid_search([1e-3, 1e-4]),
    'dropout'   : tune.grid_search([0.2, 0.3]),
}

print('Configuración del Grid Search:')
print(f'  starter     : {[32, 64, 128]}')
print(f'  activation  : ReLU, LeakyReLU, ELU, SiLU')
print(f'  optimizer   : Adam, AdamW')
print(f'  lr          : 1e-3, 1e-4')
print(f'  dropout     : 0.2, 0.3')
print()
n_combos = 3 * 4 * 2 * 2 * 2
print(f'  Total combinaciones: {n_combos}')
print()
print('▶ Ejecutando ray.tune (esto puede tardar varios minutos)...')

analysis = tune.run(
    train_tune,
    config    = search_config,
    verbose   = 0,
    resources_per_trial = {'gpu': 1 if device == 'cuda' else 0, 'cpu': 2},
)

print('\n✔  Búsqueda completada')

In [ ]:
# ── Mejor configuración encontrada ──
best_config = analysis.get_best_config(metric='accuracy', mode='max')
print('🏆 Mejor configuración encontrada:')
for k, v in best_config.items():
    print(f'   {k:<12}: {v}')

### 8.1 Visualizar resultados del grid search

Mostramos los resultados de todas las combinaciones para entender qué hiperparámetros importan más.


In [ ]:
results_df = analysis.results_df[['config/starter','config/lr',
                                     'config/activation','config/optimizer',
                                     'accuracy','val_loss']].dropna()
results_df = results_df.sort_values('accuracy', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy por starter
for s, col in zip([32, 64, 128], ['#4C9BE8','#5DBB8A','#F5A623']):
    sub = results_df[results_df['config/starter'] == s]['accuracy']
    axes[0].scatter([s]*len(sub), sub*100, color=col, alpha=0.7, s=60, label=f'starter={s}')
axes[0].set_title('Accuracy según tamaño de red (starter)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('starter'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy por lr
for lr_val, col in zip([1e-3, 1e-4], ['#AB47BC','#EF5350']):
    sub = results_df[results_df['config/lr'] == lr_val]['accuracy']
    axes[1].boxplot(sub*100, positions=[list([1e-3,1e-4]).index(lr_val)],
                   patch_artist=True,
                   boxprops=dict(facecolor=col, alpha=0.7))
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['lr=1e-3','lr=1e-4'])
axes[1].set_title('Distribución de Accuracy por Learning Rate', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)'); axes[1].grid(alpha=0.3)

plt.suptitle('Ray Tune — Resultados del Grid Search', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTop 5 configuraciones:')
print(results_df.head(5).to_string(index=False))

---
## 🚀 Paso 9 — Entrenamiento final con la mejor configuración

Ahora que `ray.tune` encontró la mejor configuración, entrenamos el modelo final  
con **más épocas** y todas las técnicas del cuadernillo **02/03/04**:

- `tqdm` para barra de progreso (cuadernillo 00)
- `ReduceLROnPlateau` — scheduler adaptativo (cuadernillo 04)
- **Early stopping** — para cuando deja de mejorar
- **Checkpoint** automático del mejor modelo (cuadernillo 04)


In [ ]:
# ── Construir modelo final con la mejor configuración ─────────
model = ModelTPS(
    input_dim  = 50,
    starter    = best_config['starter'],
    n_classes  = 4,
    activation = best_config['activation'],
    dropout    = best_config.get('dropout', 0.25)
).to(device)

criterion  = best_config['criterion']()
optimizer  = best_config['optimizer'](model.parameters(), lr=best_config['lr'],
                                       weight_decay=1e-4)
scheduler  = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                patience=5, verbose=True)

EPOCHS      = 100
PATIENCE    = 15
CHECKPOINT  = 'tps_best.pt'     # cuadernillo 04

total_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Modelo final — starter={best_config["starter"]}')
print(f'Parámetros entrenables: {total_p:,}')
print(f'Épocas máximas: {EPOCHS}  |  Early stopping: {PATIENCE}')

In [ ]:
# ── Loop de entrenamiento con tqdm + checkpoint ───────────────
history = []
best_val_loss = float('inf')
no_improve    = 0

for epoch in tqdm(range(1, EPOCHS + 1), desc='Entrenando'):
    # ── TRAIN ──
    model.train()
    train_losses = []
    for xb, yb in dl_train:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # ── EVALUATE ──
    val_acc, val_loss, _, _ = evaluate(model, dl_test, device)
    train_loss = np.mean(train_losses)
    scheduler.step(val_loss)

    # ── CHECKPOINT (cuadernillo 04) ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve    = 0
        torch.save({
            'epoch'      : epoch,
            'state_dict' : model.state_dict(),
            'optimizer'  : optimizer.state_dict(),
            'val_loss'   : val_loss,
            'val_acc'    : val_acc,
        }, CHECKPOINT)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'\n  Early stop en época {epoch}')
            break

    history.append({
        'epoch'     : epoch,
        'train_loss': train_loss,
        'val_loss'  : val_loss,
        'val_acc'   : val_acc,
        'lr'        : optimizer.param_groups[0]['lr'],
    })

# ── Cargar mejor checkpoint (cuadernillo 04) ──────────────────
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['state_dict'])
print(f'\n✔ Mejor modelo cargado — época {ckpt["epoch"]}')
print(f'  val_loss = {ckpt["val_loss"]:.4f}')
print(f'  val_acc  = {ckpt["val_acc"]:.4f} ({ckpt["val_acc"]*100:.2f}%)')

---
## 📈 Paso 10 — Visualizar el entrenamiento

Graficamos por separado (una gráfica por celda) la evolución de:
1. La función de pérdida (train vs val)
2. El accuracy de validación
3. El learning rate (schedule)

> **¿Por qué separadas?** Para que en la disertación cada gráfica tenga  
> su propia explicación y no se mezclen los conceptos.


In [ ]:
# ── 10.1 Curva de pérdida ─────────────────────────────────────
ep    = [h['epoch']      for h in history]
tr_l  = [h['train_loss'] for h in history]
val_l = [h['val_loss']   for h in history]

plt.figure(figsize=(10, 4))
plt.plot(ep, tr_l,  lw=2.5, color='#4C9BE8', label='Train Loss')
plt.plot(ep, val_l, lw=2.5, color='#E05C5C', linestyle='--', label='Val Loss')
plt.fill_between(ep, tr_l, val_l, alpha=0.08, color='gray')

# Marcar el mejor checkpoint
best_ep = ckpt['epoch']
plt.axvline(best_ep, color='green', lw=1.5, linestyle=':', label=f'Best checkpoint (ep {best_ep})')

plt.title('Curva de Pérdida — CrossEntropyLoss', fontsize=13, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Pérdida')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── 10.2 Curva de Accuracy ───────────────────────────────────
val_a = [h['val_acc']*100 for h in history]

plt.figure(figsize=(10, 4))
plt.plot(ep, val_a, lw=2.5, color='#5DBB8A', label='Val Accuracy (%)')
plt.axvline(best_ep, color='green', lw=1.5, linestyle=':', label=f'Best (ep {best_ep})')
plt.axhline(ckpt['val_acc']*100, color='orange', lw=1.2, linestyle='--',
            label=f'Mejor acc = {ckpt["val_acc"]*100:.2f}%')

plt.title('Accuracy de Validación por Época', fontsize=13, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── 10.3 Learning Rate schedule ──────────────────────────────
lrs = [h['lr'] for h in history]

plt.figure(figsize=(10, 3))
plt.plot(ep, lrs, lw=2.5, color='#F5A623')
plt.title('Learning Rate — ReduceLROnPlateau', fontsize=13, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Learning Rate')
plt.yscale('log'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print('Cuando el LR baja → el scheduler detectó que val_loss dejó de mejorar.')

---
## 🎯 Paso 11 — Evaluación final del modelo

Cargamos el mejor checkpoint y evaluamos con métricas completas:
- **Accuracy** global
- **Classification report** (precision, recall, F1 por clase)
- **Log-loss**


In [ ]:
final_acc, final_loss, preds_final, labels_final = evaluate(model, dl_test, device)
print('=' * 55)
print('           EVALUACIÓN FINAL')
print('=' * 55)
print(f'  Accuracy  : {final_acc:.4f}  ({final_acc*100:.2f}%)')
print(f'  Log-Loss  : {final_loss:.4f}')
print()
print(classification_report(labels_final, preds_final, target_names=le.classes_))

---
## 📊 Paso 12 — Métricas visuales

### 12.1 Matriz de confusión

Muestra cuántos ejemplos de cada clase **real** fueron predichos en cada clase.  
La diagonal = predicciones correctas.


In [ ]:
cm = confusion_matrix(labels_final, preds_final)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5, linecolor='white')
plt.title('Matriz de Confusión — TPS May-2021', fontsize=13, fontweight='bold')
plt.ylabel('Clase Real', fontsize=11); plt.xlabel('Clase Predicha', fontsize=11)
plt.tight_layout(); plt.show()

### 12.2 Accuracy por clase

In [ ]:
acc_por_clase = cm.diagonal() / cm.sum(axis=1)
plt.figure(figsize=(8, 4))
bars = plt.bar(le.classes_, acc_por_clase * 100,
               color=['#4C9BE8','#5DBB8A','#F5A623','#E05C5C'],
               edgecolor='white', width=0.5)
for bar, v in zip(bars, acc_por_clase):
    plt.text(bar.get_x() + bar.get_width()/2, v*100 + 0.5,
             f'{v*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.ylim(0, 110)
plt.title('Accuracy por Clase', fontsize=13, fontweight='bold')
plt.xlabel('Clase'); plt.ylabel('Accuracy (%)')
plt.axhline(final_acc*100, color='gray', lw=1.5, linestyle='--', label=f'Promedio {final_acc*100:.1f}%')
plt.legend(); plt.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

### 12.3 Distribución de probabilidades Softmax por clase

In [ ]:
model.eval()
all_probs = []
with torch.no_grad():
    for xb, _ in dl_test:
        logits = model(xb.to(device))
        all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
all_probs = np.array(all_probs)

fig, axes = plt.subplots(1, len(le.classes_), figsize=(14, 4), sharey=True)
pal = ['#4C9BE8','#5DBB8A','#F5A623','#E05C5C']
for i, (cls, col) in enumerate(zip(le.classes_, pal)):
    axes[i].hist(all_probs[:, i], bins=40, color=col, edgecolor='white', alpha=0.9)
    axes[i].set_title(f'P(clase={cls})', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Probabilidad Softmax'); axes[i].grid(alpha=0.3)
axes[0].set_ylabel('Frecuencia')
plt.suptitle('Distribución de Probabilidades Softmax por Clase', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 💾 Paso 13 — Cómo cargar el checkpoint (cuadernillo 04)

El cuadernillo **04** nos enseña que el `state_dict` guarda **solo los pesos**,  
lo que es más eficiente que guardar el modelo completo.

```python
# Guardar (ya lo hicimos durante el entrenamiento)
torch.save({'state_dict': model.state_dict(), ...}, 'tps_best.pt')

# Cargar en cualquier momento (incluso en otra sesión)
ckpt  = torch.load('tps_best.pt')
model = ModelTPS(input_dim=50, starter=..., ...)   # misma arquitectura
model.load_state_dict(ckpt['state_dict'])
model.eval()
```


In [ ]:
# Verificar el contenido del checkpoint
ckpt_info = torch.load(CHECKPOINT, map_location='cpu')
print('Contenido del checkpoint guardado:')
for k, v in ckpt_info.items():
    if k == 'state_dict':
        n_tensors = len(v)
        total_w   = sum(p.numel() for p in v.values())
        print(f'  {k:<15}: {n_tensors} tensores | {total_w:,} pesos')
    else:
        print(f'  {k:<15}: {v}')

print()
print(f'Archivo guardado en: {CHECKPOINT}')

In [ ]:
# Demostración: cargar el checkpoint y hacer una predicción
model_loaded = ModelTPS(
    input_dim  = 50,
    starter    = best_config['starter'],
    n_classes  = 4,
    activation = best_config['activation'],
).to(device)

ckpt_loaded = torch.load(CHECKPOINT, map_location=device)
model_loaded.load_state_dict(ckpt_loaded['state_dict'])
model_loaded.eval()

# Predicción sobre las primeras 5 muestras del test
x_sample = torch.tensor(X_test[:5], dtype=torch.float32).to(device)
with torch.no_grad():
    logits  = model_loaded(x_sample)
    probs   = torch.softmax(logits, dim=1)
    pred_cl = logits.argmax(dim=1).cpu().numpy()

print('Predicciones sobre 5 muestras del test:')
print(f'  Clases disponibles : {le.classes_}')
print(f'  Predicciones (idx) : {pred_cl}')
print(f'  Predicciones (lbl) : {le.inverse_transform(pred_cl)}')
print(f'  Real               : {le.inverse_transform(y_test[:5])}')
print()
print('Probabilidades Softmax:')
for i, p in enumerate(probs.cpu().numpy()):
    print(f'  muestra {i}: {dict(zip(le.classes_, [f"{v:.3f}" for v in p]))}')

---
## 🏁 Resumen — Lo que aplicamos en este cuadernillo

| Técnica | Origen | Beneficio |
|---|---|---|
| `nn.Module` con capas expandidas | Cuadernillo **00** | Control total del forward |
| `BatchNorm1d` entre capas | Cuadernillo **00** | Entrenamiento más estable |
| `tqdm` barra de progreso | Cuadernillo **00** | Visualizar entrenamiento |
| `ray.tune` grid search | Cuadernillo **00** | Mejor configuración automática |
| `Dataset` + `DataLoader` | Cuadernillo **03** | Mini-batches eficientes |
| Checkpoint `state_dict` | Cuadernillo **04** | Guardar/cargar el mejor modelo |
| `ReduceLROnPlateau` | Cuadernillo **04** | Scheduler adaptativo |
| Early Stopping | Cuadernillo **02/04** | Evitar overfitting |
| Undersampling | Lab4 original | Clases balanceadas |
| Z-score normalización | Todos | Gradiente estable |


In [ ]:
print('=' * 60)
print('           RESULTADOS FINALES — TPS May-2021')
print('=' * 60)
print(f'  Accuracy  : {final_acc*100:.2f}%')
print(f'  Log-Loss  : {final_loss:.4f}')
print(f'  Épocas    : {len(history)}  (+ early stopping)')
print(f'  Starter   : {best_config["starter"]}')
print(f'  LR inicial: {best_config["lr"]}')
print(f'  Checkpoint: {CHECKPOINT}')
print('=' * 60)